# Guide to Train Machine Learning Models on tracebloc 🚀

This notebook walks you through training an ML model on the tracebloc platform — from connecting your account to launching a training run.

**What you'll do:**
1. Connect to your tracebloc account
2. Upload a model & weights
3. Link the model with a dataset
4. Configure a training plan
5. Start training

This guide takes about **10–15 minutes** to complete.

## Prerequisites

Before you begin, make sure you have:

- ✅ A **tracebloc account** — [Sign up here](https://ai.tracebloc.io/signup) if you don't have one
- ✅ **Joined a use case** — you need an active use case with a dataset. [How to join a use case →](https://docs.tracebloc.io/join-use-case/explore-use-case)
- ✅ A **model file** (`.py`) compatible with the dataset — you can use one from the [tracebloc model zoo](https://github.com/tracebloc/model-zoo) or bring your own. [Model structure requirements →](https://docs.tracebloc.io/join-use-case/model-optimization)

📖 **Full documentation:** [docs.tracebloc.io](https://docs.tracebloc.io)

💡 **Prefer Google Colab?** [Open this guide in Colab](https://colab.research.google.com/github/tracebloc/start-training/blob/main/notebooks/traceblocTrainingGuide.ipynb) — runs entirely in your browser, no local setup needed. Once it opens, do **File → Save a copy in Drive** so your edits persist.

🐍 **Python** — the SDK declares the versions it supports in its own package metadata, and that is the only authority; this notebook does not restate it. The install cell below runs pip and, if the install fails, prints the exact range pip read together with the version you are on. A current default `python3` on macOS is often *ahead* of the supported range, so if you are running locally that is the first thing to check.

---
## 1. Connect to tracebloc

First, install the tracebloc package and log in with your tracebloc account email and password.

In [ ]:
# Install the tracebloc SDK, then explain whatever happened.
#
# This runs pip and reads pip's own answer. It deliberately does NOT check your
# Python version against a range written here first: that range would be a
# second copy of the SDK's `requires-python`, in a different repo, with nothing
# keeping the two in step. The first version of this cell did exactly that,
# went stale, and printed "OK" to a 3.14 user whose install then failed --
# a false all-clear (backend#2862, review on start-training#83).
#
# Everything below is derived from pip's output, and each branch says only what
# that output actually establishes.
import re
import subprocess
import sys

# Other extras: [sklearn] [catboost] [lightgbm] [xgboost] [lifelines]
# [scikit-survival] [all]. TensorFlow uploads were removed in SDK 1.0.0, so
# there is no [tensorflow] extra.
SPEC = "tracebloc[pytorch]>=0.14.0"
PACKAGE = "tracebloc"

running = f"{sys.version_info[0]}.{sys.version_info[1]}"
print(f"Python {sys.version.split()[0]}")
# pip's own progress is captured rather than streamed, so this cell prints
# nothing while it works. `[pytorch]` pulls ~90 packages including torch, which
# takes a few minutes on a fresh runtime -- say so, or the product's first
# onboarding cell looks like a hang (review, start-training#83).
print(f"Installing {SPEC} - ~90 packages including torch, this takes a few minutes...")

proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", SPEC],
    capture_output=True,
    text=True,
    check=False,  # the whole point of this cell is to handle a failure itself
)
output = proc.stdout + proc.stderr

if proc.returncode == 0:
    # Success side: prove it actually imports. A partial install otherwise
    # looks like success right up to the `from tracebloc import User` below.
    try:
        from importlib.metadata import version

        import tracebloc  # noqa: F401

        print(f"OK - tracebloc {version('tracebloc')} installed and imports.")
    # Broad on purpose: a half-installed dependency can raise almost anything
    # (ImportError, OSError from a bad .so, ValueError from a truncated wheel),
    # and a diagnostic cell must not become the crash it is reporting.
    except Exception as exc:
        print(f"\npip reported success but importing tracebloc did not work: {exc!r}")
        print("Restart the runtime (Runtime -> Restart session) and re-run this cell.")
        raise RuntimeError("tracebloc installed but does not import - see above") from exc
else:
    print(output[-1500:])

    # Three facts, each read out of pip's output rather than assumed.
    #
    # 1. WHICH requirement failed. pip may report `Requires-Python` for skipped
    #    *dependency* versions too, so a bound in the output is not by itself
    #    evidence about this interpreter and the SDK (Bugbot, #83).
    failed = re.search(r"No matching distribution found for (\S+)", output)
    subject = failed.group(1) if failed else ""
    is_sdk = re.match(rf"{PACKAGE}\b", subject, re.IGNORECASE) is not None

    # 2. The bound pip read, if it printed one. Entries are separated by "; "
    #    and each specifier contains commas, so the class excludes ";" and
    #    newline but NOT "," -- excluding "," captured ">=3.11" out of
    #    ">=3.11,<3.13" and hid the upper bound, which is the whole point.
    # pip does NOT sort these by version, so "the last one" is not "the newest
    # release" -- with a mixed set that reports the wrong bound (review, #83).
    # The distinct set is exact and is usually one value anyway.
    bounds = list(dict.fromkeys(b.strip() for b in re.findall(r"Requires-Python\s+([^\n;]+)", output)))

    # 3. Whether ANY version is installable here. pip lists what survived
    #    filtering as "(from versions: ...)"; a non-empty list means the
    #    interpreter is not a total wall, so "no pin can help" would be false.
    offered = re.search(r"\(from versions: ([^)]*)\)", output)
    usable = [v.strip() for v in offered.group(1).split(",")] if offered else []
    usable = [v for v in usable if v and v.lower() != "none"]

    print()
    if not bounds:
        # pip only prints the range in newer versions. Measured on pip 21.2.4
        # (the macOS system Python 3.9 default): a genuine version mismatch
        # prints just "from versions: ..." and no range at all, so claiming
        # "not a version problem" here would be wrong.
        print(
            f"pip printed no supported-Python range, so this cell cannot tell you "
            f"whether Python {running} is the cause. In order:\n"
            f"  - An older pip does not print that range. Run\n"
            f"    `{sys.executable} -m pip install --upgrade pip` and re-run this "
            "cell; a newer pip will say so.\n"
            "  - Otherwise suspect the index rather than the package: unreachable "
            "index, a custom\n    --index-url, or private-index authentication."
        )
    elif not subject:
        print(
            "pip printed a Requires-Python bound "
            f"({', '.join(bounds)}) but no 'No matching distribution found' line, "
            "so this cell\ncannot tell which requirement it belongs to. The pip "
            "output above is the authority."
        )
    elif not is_sdk:
        print(
            f"pip failed on `{subject}`, which is a dependency rather than "
            f"{PACKAGE} itself.\n"
            f"The Requires-Python bound(s) shown above ({', '.join(bounds)}) belong "
            "to that package, so\nthis is not necessarily about your interpreter. "
            "The pip output above is the authority."
        )
    else:
        print(
            f"You are on Python {running}, and the {PACKAGE} releases pip wanted "
            f"declare {', '.join(bounds)},\nso every one of them was skipped. The "
            "package is not missing; your interpreter is\noutside the range it "
            "supports."
        )
        if usable:
            print(
                f"\npip does list older releases that install here "
                f"({', '.join(usable[-3:])}). Do not reach for those:\n"
                "they predate the API this notebook uses. Use a Python inside the "
                "range instead."
            )
        else:
            print("\nNo release on the index installs on this Python, so no pin helps.")
        print(
            "  - On Colab the runtime's Python is not selectable. Please report the "
            "version above at\n"
            "    https://github.com/tracebloc/start-training/issues\n"
            "  - Locally, make a virtualenv on an interpreter inside that range and "
            "register it as\n    this notebook's kernel."
        )
    # Stop here so "Run All" halts on the cell that explains the failure. Without
    # this the run continued to `from tracebloc import User` and died with the
    # same bare ModuleNotFoundError this cell exists to replace, two cells away
    # from the explanation (Bugbot, #83). This is NOT the stale pre-install
    # assertion the review rejected -- nothing is predicted here; pip has
    # already failed.
    raise RuntimeError(f"could not install {SPEC} - see the explanation above")

In [ ]:
from tracebloc import User

# This will prompt you for your tracebloc email and password
user = User()

**Expected output:** You'll see a prompt asking for your email and password. After entering them, you should see a confirmation that you're logged in.

⚠️ **If login fails:**
- Double-check your email and password at [ai.tracebloc.io](https://ai.tracebloc.io)
- Make sure you've verified your email address
- If you don't have an account yet, [sign up here](https://ai.tracebloc.io/signup)

---
## 2. Upload model & weights file

Next, upload your model file to the platform. You have two options:

### Option A: Use a model from the tracebloc model zoo

The [tracebloc model zoo](https://github.com/tracebloc/model-zoo) has ready-to-use models for common tasks:

| Task | Framework | Path |
|------|-----------|------|
| Image classification | PyTorch / TensorFlow | `model_zoo/image_classification/` |
| Object detection | PyTorch | `model_zoo/object_detection/pytorch/` |
| Text classification | PyTorch | `model_zoo/text_classification/pytorch/` |
| Tabular classification | PyTorch / Sklearn | `model_zoo/tabular_classification/` |
| Tabular regression | PyTorch / Sklearn | `model_zoo/tabular_regression/` |
| Time series forecasting | PyTorch | `model_zoo/time_series_forecasting/pytorch/` |
| Semantic segmentation | PyTorch | `model_zoo/semantic_segmentation/pytorch/` |
| Keypoint detection | PyTorch | `model_zoo/keypoint_detection/pytorch/` |
| Time-to-event prediction | PyTorch / Lifelines / Scikit-survival | `model_zoo/time_to_event_prediction/` |

Clone the model zoo and pick a model that fits your use case:

In [ ]:
# Clone the tracebloc model zoo (skipped if it's already present)
![ -d ../model-zoo ] && echo "model-zoo already present - skipping clone" || git clone https://github.com/tracebloc/model-zoo.git ../model-zoo

In [ ]:
# List available models
!ls ../model-zoo/model_zoo/

### Option B: Use your own model

You can upload your own model file. Place your `.py` file in the working directory or provide the full path.

Make sure your model follows the [model structure requirements](https://docs.tracebloc.io/join-use-case/model-optimization).

In [ ]:
# Upload your model file to tracebloc
# Replace the path with your actual model file location
MODEL_PATH = "../model-zoo/model_zoo/image_classification/pytorch/densenet.py"  # <-- change this

user.upload_model(MODEL_PATH)

**Expected output:** A confirmation message showing the model was uploaded successfully.

💡 **Loading weights?** Follow this naming convention:
- Model file: `mymodel.py`
- Weights file: `mymodel_weights.pkl`

The weights file must be in the same directory as the model file.

```python
# To upload with pretrained weights:
user.upload_model(MODEL_PATH, weights=True)
```

---
## 3. Link uploaded model with dataset

Now connect your uploaded model to a dataset from your use case.

**Where to find the Dataset ID:**
1. Go to [ai.tracebloc.io](https://ai.tracebloc.io) and open your use case
2. Copy the ID shown next to **Dataset** in the use case panel
3. Paste it below

The dataset ID is a short alphanumeric string (e.g., `DKbtefZy`).

In [ ]:
# Paste your Dataset ID here
DATASET_ID = "YOUR_DATASET_ID_HERE"  # <-- replace with your actual dataset ID (e.g., "DKbtefZy")

training = user.link_model_dataset(DATASET_ID)

**Expected output:** A confirmation that the model and dataset are linked.

⚠️ **If this fails:**
- Make sure the dataset ID is correct (check your use case panel)
- Your model must be compatible with the dataset (e.g., an image classification model for an image dataset)

---
## 4. Set training plan

Configure your training parameters. Start by naming your experiment, then adjust any parameters you need.

| Command | Description | Example |
|---------|-------------|--------|
| `training.experiment_name("...")` | Name your experiment | `training.experiment_name("My first run")` |
| `training.epochs(n)` | Number of training epochs | `training.epochs(10)` |
| `training.optimizer("...")` | Set optimizer | `training.optimizer("adam")` |
| `training.learning_rate({...})` | Set learning rate | `training.learning_rate({"type": "constant", "value": 0.001})` |
| `training.validation_split(n)` | Validation split | `training.validation_split(0.2)` |
| `training.get_training_plan()` | View the full training plan | |
| `training.reset_training_plan()` | Reset to defaults | |

For all available parameters, see the [Hyperparameters reference](https://docs.tracebloc.io/join-use-case/hyperparameters).

In [ ]:
# Set experiment name
training.experiment_name("My Experiment")

# Set training parameters
training.epochs(10)

# Review your training plan
training.get_training_plan()

**Expected output:** A summary showing all training parameters including:
- **Training Description** — experiment name, model name, objective
- **Dataset Parameters** — dataset ID, size, classes
- **Training Parameters** — epochs, cycles, batch size, validation split
- **Hyperparameters** — optimizer, loss function, learning rate, callbacks
- **Augmentation Parameters** — data augmentation settings

Review these carefully before starting. Adjust any values using the commands in the table above.

---
## 5. Start training

Everything configured? Launch the training run:

In [ ]:
training.start()  # start the experiment as configured above

## What happens next?

Your model is now being trained on the tracebloc infrastructure. Here's what to expect:

1. **Training starts** — the model will begin training on the linked dataset inside a secure environment
2. **Monitor progress** — go to your use case on [ai.tracebloc.io](https://ai.tracebloc.io) to see training status and logs
3. **View results** — once training completes, check the leaderboard in your use case to see how your model performed
4. **Compare models** — if other team members or vendors have submitted models, you can compare performance metrics side by side

Training time depends on your dataset size, model complexity, and the number of epochs. A typical training run takes a few minutes to a few hours.

📖 **Learn more:** [How to evaluate models →](https://docs.tracebloc.io/join-use-case/model-evaluation)

---
## Logout

When you're done, log out to end your session:

In [ ]:
user.logout()

---
## Need help?

- 📖 [Documentation](https://docs.tracebloc.io)
- 📧 [support@tracebloc.io](mailto:support@tracebloc.io)
- 🐛 [Open an issue](https://github.com/tracebloc/start-training/issues)
- 💬 [Discord](https://discord.gg/tracebloc)